In [ ]:
import sys; sys.path.append('..')
import inflation, numpy as np, importlib, fd_validation, visualization, parametric_pillows, wall_generation
from numpy.linalg import norm
import MeshFEM, parallelism, benchmark, utils
import periodic_unit_helper
import numpy.linalg as la

In [ ]:
parallelism.set_max_num_tbb_threads(8)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
triArea = 0.01

In [ ]:
pts, edges = parametric_pillows.concentricCircles(8, 50)

In [ ]:
m, fuseMarkers, fuseEdges = wall_generation.triangulate_channel_walls(pts, edges, triArea)
visualization.plot_2d_mesh(m, pointList=np.where(np.array(fuseMarkers) == 1)[0], width=5, height=5)

In [ ]:
def get_average_edge_length(pts, fuseSegments):
    total_edge_length = 0
    for e in fuseSegments:
        total_edge_length += la.norm(pts[e[0]] - pts[e[1]])
    return total_edge_length / len(fuseSegments)

In [ ]:
avg_len = get_average_edge_length(m.vertices()[:, :2], fuseEdges)

In [ ]:
avg_len

In [ ]:
import igl

In [ ]:
def get_scaled_box(pts, scale = 1.1):
    cm = np.mean(np.array(pts), axis = 0)
    scaled_pts = (np.array(pts) - cm) * scale + cm
    return igl.bounding_box(scaled_pts)

In [ ]:
def get_scaled_square_box(pts, scale = 1.1):
    bbox_vx, bbox_edges = get_scaled_box(pts, scale)
    box_width = np.abs(bbox_vx[0][0] - bbox_vx[2][0])
    box_height = np.abs(bbox_vx[0][1] - bbox_vx[1][1])
    if box_width > box_height:
        height_scale = box_width / box_height
        height_mean = np.mean(bbox_vx, axis = 0)[1]
        bbox_vx[:, 1] = (bbox_vx[:, 1] - height_mean) * height_scale + height_mean
    else:
        width_scale = box_height / box_width
        width_mean = np.mean(bbox_vx, axis = 0)[0]
        bbox_vx[:, 0] = (bbox_vx[:, 0] - width_mean) * width_scale + width_mean
    return bbox_vx, bbox_edges

In [ ]:
bbox_vx, bbox_edges = get_scaled_box(pts, scale = 1.1)

bbox_vx, bbox_edges = get_scaled_square_box(pts, scale = 1.1)

box_width = np.abs(bbox_vx[0][0] - bbox_vx[2][0])
box_height = np.abs(bbox_vx[0][1] - bbox_vx[1][1])

num_width_seg = int(np.round(box_width / avg_len))
num_height_seg = int(np.round(box_height / avg_len))

top_y = bbox_vx[0][1]
right_x = bbox_vx[0][0]
bot_y = bbox_vx[3][1]
left_x = bbox_vx[3][0]

top_y,right_x, bot_y, left_x

num_width_seg, num_height_seg

bbox_vx = np.concatenate((np.linspace(bbox_vx[0], bbox_vx[1], num_height_seg),  np.linspace(bbox_vx[1], bbox_vx[3], num_width_seg)[1:], np.linspace(bbox_vx[3], bbox_vx[2], num_height_seg)[1:], np.linspace(bbox_vx[2], bbox_vx[0], num_width_seg)[1:-1]))

bbox_edges = [[i, i + 1] for i in np.arange(len(bbox_vx) - 1)] + [[len(bbox_vx) - 1, 0]]

n_vx = pts + list(bbox_vx)
n_edge = edges + list(np.array(bbox_edges) + len(pts))

In [ ]:
visualization.plot_line_segments(n_vx, n_edge)

In [ ]:
import importlib

In [ ]:
importlib.reload(parametric_pillows)

In [ ]:
from parametric_pillows import get_perioidic_mesh

In [ ]:
# n_vx

In [ ]:
m, fuseMarkers, fuseSegments = wall_generation.triangulate_channel_walls(n_vx, n_edge, triArea, flags="Y")


In [ ]:
def is_bbox(point):
    if (point[0] == left_x or point[0] == right_x or point[1] == top_y or point[1] == bot_y):
        return True
    return False

In [ ]:
finalMarkers = np.where(np.logical_and(np.array(fuseMarkers) == 1, [(not is_bbox(pt)) for pt in m.vertices()]))[0]

In [ ]:
visualization.plot_2d_mesh(m, pointList=finalMarkers, width=5, height=5)

In [ ]:
import mesh


In [ ]:
m.vertices()

In [ ]:
mesh2D = mesh.Mesh(m.vertices()[:,:2], m.elements())

In [ ]:
pc = mesh.PeriodicCondition(mesh2D)

In [ ]:
pc.identifiedNodes(0)

In [ ]:
min_x = min(igl.bounding_box(mesh2D.vertices())[0][:, 0])
max_x = max(igl.bounding_box(mesh2D.vertices())[0][:, 0])

In [ ]:
list_of_vx_pairs = []
for index, vx in enumerate(mesh2D.vertices()):
    if np.abs(vx[0] - min_x) < 1e-6:
        getPair = False
        iNs = pc.identifiedNodes(index)
        if len(iNs) == 2:
            list_of_vx_pairs.append([index, iNs[0] if iNs[0] != index else iNs[1]])
            getPair = True
        elif len(iNs) == 4:
            iNvxs = mesh2D.vertices()[iNs]
            for i, nvx in enumerate(iNvxs):
                if np.abs(nvx[0] - max_x) < 1e-6 and np.abs(nvx[1] - vx[1]) < 1e-6:
                    list_of_vx_pairs.append([index, iNs[i]])
                    getPair = True
                    break

        if getPair:
            getPair = False
        else:
            print("Warning: didn't find the matching vertex for boundary vertex: ", index)

In [ ]:
ignoreIdxs = np.array(list_of_vx_pairs)[:, 0]
mapBoundaryIdxs = dict(list_of_vx_pairs)
offset = len(mesh2D.vertices())

In [ ]:
shift = [max_x - min_x, 0]

In [ ]:
mesh2_vxs = mesh2D.vertices() + shift

In [ ]:
finalMarkers

In [ ]:
new_vertices = []
new_markers = []
reindexMesh2 = dict()
counter = 0
for i in range(len(mesh2D.vertices())):
    if i in ignoreIdxs:
        reindexMesh2[i] = mapBoundaryIdxs[i]
    else:
        reindexMesh2[i] = counter + offset
        counter += 1
        new_vertices.append(mesh2_vxs[i])


In [ ]:
new_elements = mesh2D.elements()

In [ ]:
new_elements = np.vectorize(reindexMesh2.get)(mesh2D.elements())

In [ ]:
new_elements

In [ ]:
np.array(new_vertices)

In [ ]:
np.concatenate((mesh2D.elements(), new_elements), axis = 0)

In [ ]:
np.concatenate((mesh2D.vertices(), np.array(new_vertices)), axis = 0)

In [ ]:
new_markers = np.vectorize(reindexMesh2.get)(finalMarkers)

In [ ]:
import mesh

In [ ]:
def shift_and_merge_2D_periodic_mesh(imesh, marker, axis = 0):
    mesh2D = mesh.Mesh(imesh.vertices()[:,:2], imesh.elements())
    pc = mesh.PeriodicCondition(mesh2D)
    input_vxs = mesh2D.vertices();
    # Step 1: identify the matching vertices at the stitching boundary
    min_val = min(igl.bounding_box(input_vxs)[0][:, axis])
    max_val = max(igl.bounding_box(input_vxs)[0][:, axis])
    shift = np.array([0.0, 0.0])
    shift[axis] = max_val - min_val

    list_of_vx_pairs = []
    for index, vx in enumerate(input_vxs):
        if np.abs(vx[axis] - min_val) < 1e-6:
            getPair = False
            iNs = pc.identifiedNodes(index)
            if len(iNs) == 2:
                list_of_vx_pairs.append([index, iNs[0] if iNs[0] != index else iNs[1]])
                getPair = True
            elif len(iNs) == 4:
                iNvxs = input_vxs[iNs]
                for i, nvx in enumerate(iNvxs):
                    if np.abs(nvx[axis] - max_val) < 1e-6 and np.abs(nvx[1 - axis] - vx[1 - axis]) < 1e-6:
                        list_of_vx_pairs.append([index, iNs[i]])
                        getPair = True
                        break

            if getPair:
                getPair = False
            else:
                print("Warning: didn't find the matching vertex for boundary vertex: ", index)

    # Step 2: Reindex the vertices in Mesh 2 so that the match boundary vertices are not included. 
    #        Also, shift the vertices index in Mesh 2 by the number of vertices in Mesh 1.
    #         Also, shift the vertices position in Mesh 2 by one periodic along the input axis.

    ignoreIdxs = np.array(list_of_vx_pairs)[:, 0]
    mapBoundaryIdxs = dict(list_of_vx_pairs)
    offset = len(mesh2D.vertices())
    mesh2_vxs = mesh2D.vertices() + shift

    new_vertices = []
    reindexMesh2 = dict()
    counter = 0
    for i in range(len(input_vxs)):
        if i in ignoreIdxs:
            reindexMesh2[i] = mapBoundaryIdxs[i]
        else:
            reindexMesh2[i] = counter + offset
            counter += 1
            new_vertices.append(mesh2_vxs[i])
    new_elements = np.vectorize(reindexMesh2.get)(mesh2D.elements())

    new_mesh = mesh.Mesh(np.concatenate((input_vxs, np.array(new_vertices)), axis = 0), np.concatenate((mesh2D.elements(), new_elements), axis = 0))
    new_markers = np.concatenate((marker, np.vectorize(reindexMesh2.get)(marker)), axis = 0)
    return new_mesh, new_markers


In [ ]:
nm, nmarkers = shift_and_merge_2D_periodic_mesh(m, finalMarkers, 1)

In [ ]:
visualization.plot_2d_mesh(nm, pointList=nmarkers, width=20, height=20)